## Регрессия для CC50

In [1]:
import sys
sys.path.append('../src')
from preprocessing import DatasetPreprocessor
from metrics_calculator import MetricsCalculator

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

In [2]:
df = pd.read_excel('../data/raw/classicMLCourseWorkData.xlsx')
df.head()

,Unnamed: 0,"IC50, mM","CC50, mM",SI,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,0,6.239374,175.482382,28.125000,5.094096,5.094096,0.387225,0.387225,0.417362,42.928571,...,0,0,0,0,0,0,0,0,3,0
1,1,0.771831,5.402819,7.000000,3.961417,3.961417,0.533868,0.533868,0.462473,45.214286,...,0,0,0,0,0,0,0,0,3,0
2,2,223.808778,161.142320,0.720000,2.627117,2.627117,0.543231,0.543231,0.260923,42.187500,...,0,0,0,0,0,0,0,0,3,0
3,3,1.705624,107.855654,63.235294,5.097360,5.097360,0.390603,0.390603,0.377846,41.862069,...,0,0,0,0,0,0,0,0,4,0
4,4,107.131532,139.270991,1.300000,5.150510,5.150510,0.270476,0.270476,0.429038,36.514286,...,0,0,0,0,0,0,0,0,0,0


In [3]:
# удаляем строки с пропусками
clean_df = df.dropna()
# Класс расчета метрик
metrics_helper = MetricsCalculator()

In [4]:
# таргет
y = clean_df['CC50, mM']
# признаки
X = clean_df.drop(columns=['CC50, mM'])

In [5]:
# train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

#### Линейная регрессия

In [6]:
# baseline pipeline для задачи регрессии CC50
linreg_pipeline = Pipeline([
    (
        'preprocessor',
        DatasetPreprocessor(
            target_column='CC50, mM',
            drop_unnamed=True,
            drop_leakage=True,
            drop_constant_features=True,
            drop_corr=True,
            corr_threshold=0.9,
            log_features=True,
            log_skew_threshold=2.0,
            log_min_unique_values=10
        )
    ),
    (
        'scaler',
        StandardScaler()
    ),
    (
        'model',
        LinearRegression()
    )
])

In [7]:
# обучение
linreg_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,target_column,"'CC50, mM'"
,drop_unnamed,True
,drop_leakage,True
,drop_constant_features,True
,drop_corr,True
,corr_threshold,0.9
,log_features,True


In [8]:
# предсказания
linreg_y_train_pred = linreg_pipeline.predict(X_train)
linreg_y_test_pred = linreg_pipeline.predict(X_test)

In [9]:
# метрики
linreg_train_metrics = metrics_helper.regression_metrics(y_train, linreg_y_train_pred)
linreg_test_metrics = metrics_helper.regression_metrics(y_test, linreg_y_test_pred)

print('Метрики на train:')
display(linreg_train_metrics)

print('Метрики на test:')
display(linreg_test_metrics)

Метрики на train:


{'MAE': 267.1331006043024,
 'MSE': 140068.15100015057,
 'RMSE': np.float64(374.25679820165),
 'R2': 0.6465765975715223}

Метрики на test:


{'MAE': 4472.712343132247,
 'MSE': 3402723391.547239,
 'RMSE': np.float64(58332.867163780305),
 'R2': -7222.276359161034}

- Модель показывает хорошее качество на train ($R^2 \approx 0.65$), но крайне плохое на test ($R^2 < 0$)
- Наблюдается сильное переобучение (большой разрыв между train и test)
- Линейная модель не способна корректно описать зависимость между признаками и таргетом
- Вероятно, зависимости в данных носят нелинейный характер

**Необходимо:**
- попробовать логарифмирование таргета
- далее использовать более сложные модели (RandomForest, GradientBoosting)

In [10]:
# pipeline с логарифмированием таргета
loglin_pipeline = Pipeline([
    (
        'preprocessor',
        DatasetPreprocessor(
            target_column='CC50, mM',
            drop_unnamed=True,
            drop_leakage=True,
            drop_constant_features=True,
            drop_corr=True,
            corr_threshold=0.9,
            log_features=True,
            log_skew_threshold=2.0,
            log_min_unique_values=10
        )
    ),
    ('scaler', StandardScaler()),
    (
        'model',
        TransformedTargetRegressor(
            regressor=LinearRegression(),
            func=np.log1p,
            inverse_func=np.expm1
        )
    )
])

In [11]:
# обучение
loglin_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,target_column,"'CC50, mM'"
,drop_unnamed,True
,drop_leakage,True
,drop_constant_features,True
,drop_corr,True
,corr_threshold,0.9
,log_features,True


In [12]:
# предсказания
loglin_y_train_pred = loglin_pipeline.predict(X_train)
loglin_y_test_pred = loglin_pipeline.predict(X_test)

In [13]:
loglin_train_metrics = metrics_helper.regression_metrics(y_train, loglin_y_train_pred)
loglin_test_metrics = metrics_helper.regression_metrics(y_test, loglin_y_test_pred)

print('Метрики на train:')
display(loglin_train_metrics)

print('Метрики на test:')
display(loglin_test_metrics)

Метрики на train:


{'MAE': 358.65104919799046,
 'MSE': 1434315.063224857,
 'RMSE': np.float64(1197.6289338625954),
 'R2': -2.6190990327187307}

Метрики на test:


{'MAE': 741.1604075819782,
 'MSE': 11041628.97881947,
 'RMSE': np.float64(3322.8946686314735),
 'R2': -22.43908933869238}

- Логарифмирование таргета не улучшило качество модели
- На train наблюдается плохое качество ($R^2 < 0$), на test также отрицательное значение ($R^2 \ll 0$)
- Модель не способна адекватно описывать зависимость даже на обучающей выборке


#### Случайный лес

In [14]:
rf_pipeline = Pipeline([
    (
        'preprocessor',
        DatasetPreprocessor(
            target_column='CC50, mM',
            drop_unnamed=True,
            drop_leakage=True,
            drop_constant_features=True,
            drop_corr=True,
            corr_threshold=0.9,
            log_features=True,
            log_skew_threshold=2.0,
            log_min_unique_values=10
        )
    ),
    ('scaler', StandardScaler()),
    (
        'model',
        RandomForestRegressor(
            n_estimators=100,
            max_depth=None,
            random_state=42,
            n_jobs=-1
        )
    )
])

# обучение
rf_pipeline.fit(X_train, y_train)

# предсказания
rf_y_train_pred = rf_pipeline.predict(X_train)
rf_y_test_pred = rf_pipeline.predict(X_test)

# метрики
rf_train_metrics = metrics_helper.regression_metrics(y_train, rf_y_train_pred)
rf_test_metrics = metrics_helper.regression_metrics(y_test, rf_y_test_pred)

print('Метрики на train:')
display(rf_train_metrics)

print('Метрики на test:')
display(rf_test_metrics)

Метрики на train:


{'MAE': 90.38736108314306,
 'MSE': 19764.04533454385,
 'RMSE': np.float64(140.5846554021592),
 'R2': 0.9501308748776325}

Метрики на test:


{'MAE': 254.4190024308343,
 'MSE': 230888.70030086808,
 'RMSE': np.float64(480.50879315665816),
 'R2': 0.5098711536107756}

- Модель показывает высокое качество на train ($R^2 \approx 0.95$), но заметно хуже на test ($R^2 \approx 0.51$)

**Выполнить подбор гиперпараметров RandomForest**

In [15]:
# сетка гиперпараметров
rf_param_grid = {
    'model__n_estimators': [200, 300],
    'model__max_depth': [5, 8, 10, 12],
    'model__min_samples_split': [5, 10, 15, 20],
    'model__min_samples_leaf': [2, 4, 6, 8],
    'model__max_features': ['sqrt', 'log2', 0.3, 0.5]
}

# подбор параметров
rf_grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    scoring='r2',
    cv=5,
    n_jobs=-1,
    verbose=1
)

# обучение
rf_grid_search.fit(X_train, y_train)

# лучшая модель
best_rf_pipeline = rf_grid_search.best_estimator_

# предсказания
best_rf_y_train_pred = best_rf_pipeline.predict(X_train)
best_rf_y_test_pred = best_rf_pipeline.predict(X_test)

# метрики
best_rf_train_metrics = metrics_helper.regression_metrics(y_train, best_rf_y_train_pred)
best_rf_test_metrics = metrics_helper.regression_metrics(y_test, best_rf_y_test_pred)

print('Лучшие параметры:')
print(rf_grid_search.best_params_)

print('Метрики на train:')
display(best_rf_train_metrics)

print('Метрики на test:')
display(best_rf_test_metrics)

Fitting 5 folds for each of 512 candidates, totalling 2560 fits
Лучшие параметры:
{'model__max_depth': 10, 'model__max_features': 0.5, 'model__min_samples_leaf': 2, 'model__min_samples_split': 5, 'model__n_estimators': 200}
Метрики на train:


{'MAE': 124.73732232592776,
 'MSE': 35631.18003374998,
 'RMSE': np.float64(188.76223148116782),
 'R2': 0.9100945304828361}

Метрики на test:


{'MAE': 255.1232556784414,
 'MSE': 224382.0402194219,
 'RMSE': np.float64(473.68981435051137),
 'R2': 0.5236834441014334}

- Модель показывает высокое качество на train ($R^2 \approx 0.91$), на test качество остаётся на уровне ($R^2 \approx 0.52$)

**Вывод:** подбор гиперпараметров позволил немного снизить переобучение (качество на train уменьшилось), однако качество на test практически не изменилось

#### Градиентный бустинг

In [16]:
# pipeline
gb_pipeline = Pipeline([
    (
        'preprocessor',
        DatasetPreprocessor(
            target_column='CC50, mM',
            drop_unnamed=True,
            drop_leakage=True,
            drop_constant_features=True,
            drop_corr=True,
            corr_threshold=0.9,
            log_features=True,
            log_skew_threshold=2.0,
            log_min_unique_values=10
        )
    ),
    ('scaler', StandardScaler()),
    (
        'model',
        GradientBoostingRegressor(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=42
        )
    )
])

# обучение
gb_pipeline.fit(X_train, y_train)

# предсказания
gb_y_train_pred = gb_pipeline.predict(X_train)
gb_y_test_pred = gb_pipeline.predict(X_test)

# метрики
gb_train_metrics = metrics_helper.regression_metrics(y_train, gb_y_train_pred)
gb_test_metrics = metrics_helper.regression_metrics(y_test, gb_y_test_pred)

print('Метрики на train:')
display(gb_train_metrics)

print('Метрики на test:')
display(gb_test_metrics)

Метрики на train:


{'MAE': 152.5378864901245,
 'MSE': 45143.6439223627,
 'RMSE': np.float64(212.47033657045566),
 'R2': 0.8860924477182258}

Метрики на test:


{'MAE': 279.61591776149123,
 'MSE': 231360.56180897966,
 'RMSE': np.float64(480.9995444997632),
 'R2': 0.5088694894482382}

- Модель показывает высокое качество на train ($R^2 \approx 0.89$), на test качество ниже ($R^2 \approx 0.51$)

**Вывод:** GradientBoosting показывает сопоставимое качество с RandomForest, при этом разрыв между train и test остаётся

In [17]:
# сетка гиперпараметров
gb_param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__learning_rate': [0.01, 0.03, 0.1],
    'model__max_depth': [2, 3, 4],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4]
}

# подбор параметров
gb_grid_search = GridSearchCV(
    estimator=gb_pipeline,
    param_grid=gb_param_grid,
    scoring='r2',
    cv=5,
    n_jobs=-1,
    verbose=1
)

# обучение
gb_grid_search.fit(X_train, y_train)

# лучшая модель
best_gb_pipeline = gb_grid_search.best_estimator_

# предсказания
best_gb_y_train_pred = best_gb_pipeline.predict(X_train)
best_gb_y_test_pred = best_gb_pipeline.predict(X_test)

# метрики
best_gb_train_metrics = metrics_helper.regression_metrics(y_train, best_gb_y_train_pred)
best_gb_test_metrics = metrics_helper.regression_metrics(y_test, best_gb_y_test_pred)

print('Лучшие параметры:')
print(gb_grid_search.best_params_)

print('Метрики на train:')
display(best_gb_train_metrics)

print('Метрики на test:')
display(best_gb_test_metrics)

Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Лучшие параметры:
{'model__learning_rate': 0.03, 'model__max_depth': 4, 'model__min_samples_leaf': 2, 'model__min_samples_split': 2, 'model__n_estimators': 300}
Метрики на train:


{'MAE': 114.27645648426004,
 'MSE': 25759.653991498,
 'RMSE': np.float64(160.49814326495493),
 'R2': 0.9350026077016912}

Метрики на test:


{'MAE': 266.6903963170056,
 'MSE': 230723.02755422847,
 'RMSE': np.float64(480.3363691770887),
 'R2': 0.510222842507126}

- Модель показывает высокое качество на train ($R^2 \approx 0.94$), на test качество остаётся на уровне ($R^2 \approx 0.51$)

**Вывод:** подбор гиперпараметров не привёл к улучшению качества на test, несмотря на высокое качество на train

- Все модели (LinearRegression, RandomForest, GradientBoosting) показывают схожее качество на test ($R^2 \approx 0.5$)
- Подбор гиперпараметров не приводит к существенному улучшению качества
- Лучший результат показала модель RandomForest ($R^2 \approx 0.52$ на test)

**Вывод:** RandomForest - лучшая модель для данной задачи из опробованных